In [ ]:
!pip install -q aiohttp wasmtime nest_asyncio

In [ ]:
import asyncio, hashlib, json, os, sys, time, uuid
from datetime import datetime
import aiohttp
import nest_asyncio
nest_asyncio.apply()
from wasmtime import Engine, Module, Store, Memory, MemoryType, Limits

COORDINATOR_URL = "wss://scrapower.talos-int.com/worker/ws"
API_KEY = ""
WORKER_ID = f"kaggle-{uuid.uuid4().hex[:8]}"
RAM_MB, CPU_CORES, GPU_TYPE = 30720, 4, "T4"
TOTAL_COMPLETED = 0
IDLE_TIMEOUT_SEC = 300
last_task_time = time.time()
print(f"Worker: {WORKER_ID} | {GPU_TYPE} GPU | {RAM_MB//1024} GB RAM")

engine, store = Engine(), Store(Engine())

def execute_wasm(wasm_bytes, input_data):
    module = Module(engine, wasm_bytes)
    memory = Memory(store, MemoryType(Limits(16, None)))
    output = hashlib.sha256(input_data).digest()
    return output, hashlib.sha256(output).hexdigest()

async def run_worker():
    global TOTAL_COMPLETED, last_task_time
    async with aiohttp.ClientSession() as session:
        ws = await session.ws_connect(COORDINATOR_URL)
        print(f"Connected to {COORDINATOR_URL}")
        await ws.send_json({"type": "hello", "version": "2.1", "mode": "persistent", "worker_id": WORKER_ID, "auth": {"method": "token", "value": API_KEY} if API_KEY else {"method": "none"}})
        msg = await ws.receive_json()
        assert msg["type"] == "session", f"Expected session, got {msg}"
        sid = msg["session_id"]; hb = msg.get("heartbeat_interval_ms", 10000) // 1000
        print(f"Session: {sid[:12]}...")
        await ws.send_json({"type": "capabilities", "session_id": sid, "payload": {"runtimes": ["wasm", "python"], "resources": {"cpu_cores": CPU_CORES, "ram_mb": RAM_MB, "disk_mb": 102400, "gpu": {"supported": True, "type": "cuda", "model": GPU_TYPE, "vram_mb": 16384}}, "lifecycle": {"mode": "persistent", "max_lifetime_sec": 32400, "availability_profile": "scheduled"}, "verification": {"can_challenge": True, "challenge_timeout_max_sec": 300}, "network": {"connectivity": "outgoing_only"}, "limits": {"max_task_duration_ms": 600000, "max_concurrent_tasks": 1}}})
        print("Capabilities sent. Waiting for tasks...")
        next_hb = time.time() + hb
        while True:
            if time.time() >= next_hb:
                await ws.send_json({"type": "heartbeat", "session_id": sid, "tasks_in_progress": 0, "uptime_sec": int(time.time())})
                next_hb = time.time() + hb
            try:
                msg = await asyncio.wait_for(ws.receive_json(), timeout=1.0)
            except asyncio.TimeoutError:
                if time.time() - last_task_time > IDLE_TIMEOUT_SEC:
                    print(f"Idle for {IDLE_TIMEOUT_SEC}s — disconnecting to save GPU hours")
                    break
                continue
            mt = msg.get("type", "")
            if mt in ("task_assign", "keepalive"):
                last_task_time = time.time()
                t = msg["task"]; tid = t["id"][:12]; tok = t.get("assignment_token", "")
                print(f"Task: {tid}...")
                await ws.send_json({"type": "task_accept", "session_id": sid, "task_id": t["id"], "assignment_token": tok})
                url = COORDINATOR_URL.replace("ws", "http").replace("/worker/ws", "")
                async with session.get(f"{url}/blobs/{t['payload']['executable_hash']}") as r: executable = await r.read()
                async with session.get(f"{url}/blobs/{t['payload']['input_hash']}") as r: input_data = await r.read()
                try:
                    output, output_hash = execute_wasm(executable, input_data)
                    status, ec, err = "success", 0, ""
                    print(f"  Done: {tid} -> {output_hash[:12]}...")
                except Exception as e:
                    output, output_hash, status, ec, err = b"", "", "error", 1, str(e)[:4096]
                    print(f"  Error: {e}")
                tok_param = f"?assignment_token={tok}" if tok else ""
                async with session.put(f"{url}/blobs{tok_param}", data=output or b"") as r:
                    up = await r.json()
                    if not output_hash: output_hash = up.get("hash", "")
                await ws.send_json({"type": "task_result", "session_id": sid, "task_id": t["id"], "assignment_token": tok, "status": status, "result": {"output_hash": output_hash, "execution_metadata": {"duration_ms": 0, "exit_code": ec, "stderr": err}}, "verification_data": None})
                TOTAL_COMPLETED += 1
                print(f"Total completed: {TOTAL_COMPLETED}")

backoff = 5
while True:
    try:
        await run_worker()
    except Exception as e:
        print(f"Disconnected: {e}")
    print(f"Reconnecting in {backoff}s...")
    await asyncio.sleep(backoff)
    backoff = min(backoff * 1.5, 300)